# ReservoirNet -- Tier-3 GPU runs (ML baselines, El Nino folds, level model)

Run this **after** `kaggle_runner.ipynb` has produced `runs/seed*/` artifacts, or
standalone: it clones the repo and produces the CPU-prohibitive analyses.

**What it does (all GPU):**
1. ML baselines at full strength, LSTM seed-averaged — ARIMA, Random Forest, LSTM.
2. Rolling-origin folds **2015 + 2016** (the severe El Nino the design requires)
   plus 2023 (recent El Nino onset).
3. El Nino stratified evaluation + the rule-curve self-contained level model.

Design notes that matter:
* The LSTM is **not** speed-capped: lookback 90 (matches the GNN), 2 layers,
  hidden 64, early stopping. `--train-stride 3` only drops redundant overlapping
  windows (89/90 days shared), it does not reduce capacity.
* Single-seed LSTM skill is unstable on this task, so it is averaged over 3 seeds.


## 1. Setup


In [ ]:
# Get the code (public repo) + verify it is the fixed revision.
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rajhodedara/gnn-reservoirnet.git"
dst = Path("/kaggle/working/gnn-reservoirnet")
src = next((c for c in ["/kaggle/input/gnn-reservoirnet",
                        "/kaggle/input/gnn-reservoirnet-dataset"] if Path(c).exists()), None)
if dst.exists():
    shutil.rmtree(dst)
if src:
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints"))
    print("Copied attached input:", src)
else:
    r = subprocess.run(["git", "clone", REPO_URL, str(dst)], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
    print("Cloned:", REPO_URL)

os.chdir(dst)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("HEAD:", head)

# Guard: the scripts this notebook calls must exist.
for f in ["scripts/run_ml_baselines.py", "scripts/el_nino_analysis.py",
          "scripts/eval_levels_rulecurve.py", "scripts/rolling_origin_eval.py",
          "scripts/analyse_el_nino_folds.py"]:
    assert Path(f).exists(), f"missing {f} -- wrong revision"
print("All required scripts present.")


In [ ]:
# Install what Kaggle lacks. torch is preinstalled (GPU build).
!pip install -q torch-geometric properscoring captum statsmodels
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
# Sanity: dataset present; locate GNN predictions (may be absent in a fresh clone).
import glob, pandas as pd
from pathlib import Path

files = sorted(glob.glob("data/raw/wris_v2/*.csv"))
assert len(files) == 10, f"expected 10 reservoir CSVs, found {len(files)}"
cli = pd.read_csv("data/raw/enso/combined_climate_indices.csv")
assert "iod" in cli.columns, "IOD missing in climate CSV"
print("Data OK:", len(files), "reservoirs")

def find_predictions():
    cands = sorted(Path("runs").glob("seed*/predictions_test.npz"))
    if cands:
        return cands[0].parent
    if Path("runs/predictions_test.npz").exists():
        return Path("runs")
    return None

pred_dir = find_predictions()
print("GNN predictions:", pred_dir if pred_dir else "NOT FOUND (will train one seed later)")


## 2. ML baselines (full strength, GPU)

ARIMA/SARIMA, Random Forest, and a **seed-averaged** LSTM on the same target and
splits as the GNN. Writes `outputs/ml_baselines.json`.


In [ ]:
!python scripts/run_ml_baselines.py --config configs/default_config.yaml --device cuda --lstm-seeds 3 --epochs 200


## 3. Severe-El-Nino rolling-origin folds

Fold Y: test=Y, val=Y-1, train <= (Y-2)-12-31. 2015 is the strongest El Nino on
record (JJAS ONI +1.74); 2023 is the recent onset.


In [ ]:
for yr in [2015, 2016, 2023]:
    print(f"===== FOLD {yr} =====")
    !python scripts/rolling_origin_eval.py --fold {yr}
!python scripts/aggregate_rolling_origin.py
!python scripts/analyse_el_nino_folds.py --rolling-dir outputs/rolling_origin


## 4. El Nino stratified evaluation + rule-curve level model


In [ ]:
# Ensure GNN predictions exist for the El Nino / level analyses.
# A fresh clone has no runs/ (git-ignored), so train ONE seed if needed.
from pathlib import Path

if pred_dir is None:
    print("No predictions found -> training a single seed (~3-5 min on T4)...")
    !python main.py --config configs/default_config.yaml --seed 42 --output-dir runs/seed42
    pred_dir = Path("runs/seed42")
    assert (pred_dir / "predictions_test.npz").exists(), "training did not produce predictions"
print("Using predictions from:", pred_dir)


In [ ]:
# El Nino stratified evaluation + rule-curve level model.
# pred_dir was resolved (and trained if necessary) in the previous cell.
print("Using predictions from:", pred_dir)
!python scripts/el_nino_analysis.py --pred-dir {pred_dir} --out-dir outputs/el_nino
!python scripts/eval_levels_rulecurve.py --pred-dir {pred_dir} --out outputs/level_rulecurve


## 5. Package outputs for download


In [ ]:
import shutil
from pathlib import Path
for folder in ["runs", "outputs"]:
    src = Path(folder)
    if src.exists() and any(src.iterdir()):
        z = shutil.make_archive(f"artifacts_{folder}", "zip", root_dir=".", base_dir=folder)
        print("created", z, f"({Path(z).stat().st_size/1e6:.1f} MB)")
print("\nDownload artifacts_runs.zip / artifacts_outputs.zip from the Output tab.")
